In [1224]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,OrdinalEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

In [1225]:
df=pd.read_csv('/kaggle/input/datasets/harshadapatil31/student-performance-and-study-habits-dataset/student_performance_dataset.csv')

In [1226]:
df.head()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [1227]:
df=df.drop('student_id',axis=True)

In [1228]:
df['internet_access']=df['internet_access'].map({'Yes':1 ,'No':0})
df['extracurricular_activities']=df['extracurricular_activities'].map({'Yes':1 ,'No':0})
df['part_time_job']=df['part_time_job'].map({'Yes':1 ,'No':0})

In [1229]:
df.head()

,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,Male,4.0,98.0,6.5,Bachelors,1,1,0,76.9,100.0,A
1,Female,6.3,100.0,5.7,High School,1,1,1,75.5,100.0,A
2,Male,4.9,85.3,7.9,Bachelors,1,0,1,88.5,97.3,A
3,Male,2.6,77.5,8.0,NaN,1,1,0,85.1,83.8,B
4,Male,2.2,89.6,4.6,Bachelors,1,0,1,61.8,68.3,D


In [1230]:
df.isna().sum()

gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                0
final_grade                     0
dtype: int64

In [1231]:
df['parental_education']=df['parental_education'].fillna(df['parental_education'].mode()[0])

In [1232]:
df.isna().sum()

gender                        0
study_time_hours              0
attendance_percent            0
sleep_hours                   0
parental_education            0
internet_access               0
extracurricular_activities    0
part_time_job                 0
previous_grade                0
final_exam_score              0
final_grade                   0
dtype: int64

In [1233]:
df.head()

,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,Male,4.0,98.0,6.5,Bachelors,1,1,0,76.9,100.0,A
1,Female,6.3,100.0,5.7,High School,1,1,1,75.5,100.0,A
2,Male,4.9,85.3,7.9,Bachelors,1,0,1,88.5,97.3,A
3,Male,2.6,77.5,8.0,High School,1,1,0,85.1,83.8,B
4,Male,2.2,89.6,4.6,Bachelors,1,0,1,61.8,68.3,D


In [1234]:
lbl=LabelEncoder()
df['final_grade']=lbl.fit_transform(df['final_grade'])

In [1235]:
df.head()

,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,Male,4.0,98.0,6.5,Bachelors,1,1,0,76.9,100.0,0
1,Female,6.3,100.0,5.7,High School,1,1,1,75.5,100.0,0
2,Male,4.9,85.3,7.9,Bachelors,1,0,1,88.5,97.3,0
3,Male,2.6,77.5,8.0,High School,1,1,0,85.1,83.8,1
4,Male,2.2,89.6,4.6,Bachelors,1,0,1,61.8,68.3,3


In [1236]:
X=df.drop('final_grade',axis=1)
y=df['final_grade']

In [1237]:
Categorical_columns = ['gender']
Ordinal_columns = ['parental_education']
numeric_columns = [feature for feature in X if X[feature].dtype=='int' or X[feature].dtype=='float']
print(numeric_columns)

['study_time_hours', 'attendance_percent', 'sleep_hours', 'internet_access', 'extracurricular_activities', 'part_time_job', 'previous_grade', 'final_exam_score']


In [1238]:
ohe=OneHotEncoder()
oe=OrdinalEncoder(categories=[['High School','Bachelors','Masters','PhD']])
scaler=StandardScaler()

In [1239]:
preprocessor = ColumnTransformer([
    ("OneHotEncoder",ohe,Categorical_columns),
    ("OrdinalEncoder",oe,Ordinal_columns),
    ("StandardScaler",scaler,numeric_columns)
],remainder='passthrough')

In [1240]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [1241]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [1242]:
X_train

array([[ 1.        ,  0.        ,  2.        , ..., -0.70181003,
        -0.87807664,  0.3259011 ],
       [ 0.        ,  1.        ,  0.        , ..., -0.70181003,
        -1.00780856,  1.45003214],
       [ 0.        ,  1.        ,  0.        , ...,  1.42488702,
         0.8814126 ,  0.2580656 ],
       ...,
       [ 1.        ,  0.        ,  0.        , ..., -0.70181003,
         1.27871662,  0.69415092],
       [ 1.        ,  0.        ,  0.        , ...,  1.42488702,
         0.54897454,  0.91703897],
       [ 1.        ,  0.        ,  1.        , ..., -0.70181003,
        -0.35104069,  0.53909836]])

In [1243]:
y_train

541    1
440    0
482    1
422    2
778    1
      ..
106    2
270    2
860    0
435    0
102    1
Name: final_grade, Length: 700, dtype: int64

In [1244]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128,activation='relu'),
    Dropout(0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(5,activation='softmax')
])

In [1245]:
model.summary()

Model: "sequential_21"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_68 (Dense)                │ (None, 128)            │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_69 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_70 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,117 (39.52 KB)

 Trainable params: 10,117 (39.52 KB)

 Non-trainable params: 0 (0.00 B)

In [1246]:
model.compile(optimizer='Adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [1247]:
early_stopping_callback = EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [1248]:
history=model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=30,batch_size=32,callbacks=[early_stopping_callback])

Epoch 1/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3043 - loss: 1.5159 - val_accuracy: 0.5000 - val_loss: 1.2932
Epoch 2/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4757 - loss: 1.2419 - val_accuracy: 0.6267 - val_loss: 1.0982
Epoch 3/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5543 - loss: 1.0773 - val_accuracy: 0.6600 - val_loss: 0.9274
Epoch 4/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6486 - loss: 0.9120 - val_accuracy: 0.7000 - val_loss: 0.7991
Epoch 5/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6886 - loss: 0.7978 - val_accuracy: 0.7300 - val_loss: 0.7048
Epoch 6/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7286 - loss: 0.7089 - val_accuracy: 0.7600 - val_loss: 0.6298
Epoch 7/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7600 - loss: 0.6749 - val_accuracy: 0.7733 - val_loss: 0.5687
Epoch 8/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7757 - loss: 0.6080 - val_accuracy: 0.7833 - val_loss